# 01 · Smoke Test de Fine-Tuning NMT (MarianMT)

**Proyecto:** Traducción de Mensajes de Audio con Clonación de Voz mediante IA  
**Módulo:** Fine-Tuning del Modelo de Traducción (NMT)  
**Arquitectura:** `Helsinki-NLP/opus-mt-es-en` adaptado con `Seq2SeqTrainer`  

> **AVISO IMPORTANTE:** Este notebook ejecuta exclusivamente una **CONFIGURACIÓN DE DEBUG / SMOKE TEST** (256 pares de entrenamiento, 64 de evaluación, 1 época). Su objetivo es verificar que la tokenización, el cálculo de gradientes con AdamW, la precisión mixta fp16, la evaluación con generación de haz y el guardado de métricas operan sin errores ni problemas de memoria (OOM). **NO reemplaza el entrenamiento completo sobre 100 000 pares.**

## 1. Configuración del Entorno y GPU

In [1]:
# 1. Instalar dependencias si se ejecuta en Google Colab
%pip install --quiet "transformers>=4.38.0" "datasets>=3.0.0" "accelerate" "sacrebleu" "evaluate" "sentencepiece" "sacremoses" "pyyaml"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.7 MB/s eta 0:00:00


In [2]:
# 2. Directorios y entorno de trabajo
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/Setrox1250/Proyecto-IA-Traductor.git"
REPO_DIR = Path("/content/Proyecto-IA-Traductor")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    PROJECT_ROOT = REPO_DIR
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / (1024**2):.2f} MB")
else:
    print("GPU no disponible. Ejecutando en CPU.")
    print("TIP Colab: Se recomienda activar GPU en Colab (Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU) para completar el smoke test en ~10 segundos.")

CUDA disponible: True
GPU: Tesla T4
VRAM total: 14912.69 MB


## 2. Ejecutar Fine-Tuning Debug (Smoke Test — 256 Train / 64 Eval / 1 Época)

In [3]:
import time
import json
import inspect
import unicodedata
import numpy as np
import pandas as pd
import evaluate
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    return " ".join(text.split()).strip()

def load_smoke_data(project_root: Path, train_limit: int = 256, eval_limit: int = 64) -> DatasetDict:
    train_csv = project_root / "data/processed/translation/train.csv"
    val_csv = project_root / "data/processed/translation/validation.csv"
    test_csv = project_root / "data/processed/translation/test.csv"

    if train_csv.exists() and val_csv.exists() and test_csv.exists():
        print("Cargando muestras desde archivos CSV locales...")
        train_df = pd.read_csv(train_csv, nrows=train_limit)
        val_df = pd.read_csv(val_csv, nrows=eval_limit)
        test_df = pd.read_csv(test_csv, nrows=eval_limit)
        return DatasetDict({
            "train": Dataset.from_pandas(train_df[["text_es", "text_en"]]),
            "validation": Dataset.from_pandas(val_df[["text_es", "text_en"]]),
            "test": Dataset.from_pandas(test_df[["text_es", "text_en"]]),
        })

    print("Archivos locales no encontrados; cargando muestras desde Hugging Face (streaming)...")
    opus_stream = load_dataset("Helsinki-NLP/opus-100", "en-es", streaming=True)

    def _extract_pairs(split_stream, limit: int):
        rows = []
        for item in split_stream:
            trans = item.get("translation") or {}
            es = normalize_text(trans.get("es"))
            en = normalize_text(trans.get("en"))
            if es and en:
                rows.append({"text_es": es, "text_en": en})
                if len(rows) >= limit:
                    break
        return rows

    train_rows = _extract_pairs(opus_stream["train"], train_limit)
    val_rows = _extract_pairs(opus_stream["validation"], eval_limit)
    test_rows = _extract_pairs(opus_stream["test"], eval_limit)

    return DatasetDict({
        "train": Dataset.from_list(train_rows),
        "validation": Dataset.from_list(val_rows),
        "test": Dataset.from_list(test_rows),
    })

def run_smoke_fine_tuning(
    project_root: Path = PROJECT_ROOT,
    model_name: str = "Helsinki-NLP/opus-mt-es-en",
    max_length: int = 128,
    train_samples: int = 256,
    eval_samples: int = 64,
) -> dict:
    is_cuda = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if is_cuda else "CPU"
    print(f"Dispositivo de ejecución: {device_name} (CUDA={is_cuda})")
    print(f"Smoke test: {train_samples} train, {eval_samples} eval, max_length={max_length}, 1 epoch")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    raw_datasets = load_smoke_data(project_root, train_limit=train_samples, eval_limit=eval_samples)
    checkpoint_dir = project_root / "models/checkpoints/debug"
    output_dir = project_root / "outputs/smoke_tests/nmt_training"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    def preprocess_function(examples):
        model_inputs = tokenizer(examples["text_es"], max_length=max_length, truncation=True, padding=False)
        labels = tokenizer(text_target=examples["text_en"], max_length=max_length, truncation=True, padding=False)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized_datasets = raw_datasets.map(
        preprocess_function,
        batched=True,
        remove_columns=raw_datasets["train"].column_names,
        desc="Tokenizando particiones"
    )

    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=8 if is_cuda else None,
        label_pad_token_id=-100,
    )

    bleu_metric = evaluate.load("sacrebleu")
    chrf_metric = evaluate.load("chrf")

    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        decoded_preds = [p.strip() for p in tokenizer.batch_decode(preds, skip_special_tokens=True)]
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = [[l.strip()] for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]

        b = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
        c = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)
        return {
            "bleu": round(float(b["score"]), 2),
            "chrf": round(float(c["score"]), 2),
        }

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(checkpoint_dir),
        eval_strategy="steps",
        eval_steps=16,
        learning_rate=5e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        save_strategy="no",
        num_train_epochs=1.0,
        predict_with_generate=True,
        generation_max_length=max_length,
        generation_num_beams=4,
        fp16=is_cuda,
        warmup_steps=3,
        label_smoothing_factor=0.1,
        logging_steps=8,
        seed=42,
        report_to="none",
    )

    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": tokenized_datasets["train"],
        "eval_dataset": tokenized_datasets["validation"],
        "data_collator": collator,
        "compute_metrics": compute_metrics,
    }
    sig = inspect.signature(Seq2SeqTrainer.__init__)
    if "processing_class" in sig.parameters:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sig.parameters:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Seq2SeqTrainer(**trainer_kwargs)

    if is_cuda:
        torch.cuda.reset_peak_memory_stats(0)

    print("Evaluando modelo BASE antes del fine-tuning (smoke test baseline)...")
    base_eval_metrics = trainer.evaluate()
    base_bleu = base_eval_metrics.get("eval_bleu", 0.0)
    base_chrf = base_eval_metrics.get("eval_chrf", 0.0)
    base_loss = round(float(base_eval_metrics.get("eval_loss", 0.0)), 4)
    print(f"Base Metrics -> Loss: {base_loss} | BLEU: {base_bleu} | chrF: {base_chrf}")

    start_time = time.time()
    print("Iniciando entrenamiento del smoke test...")
    train_result = trainer.train()
    training_duration = time.time() - start_time
    print(f"Entrenamiento completado en {training_duration:.2f} s")

    print("Evaluando modelo FINE-TUNED (smoke test)...")
    eval_metrics = trainer.evaluate()
    debug_bleu = eval_metrics.get("eval_bleu", 0.0)
    debug_chrf = eval_metrics.get("eval_chrf", 0.0)
    debug_loss = round(float(eval_metrics.get("eval_loss", 0.0)), 4)

    peak_vram_mb = (
        round(torch.cuda.max_memory_allocated(0) / (1024 * 1024), 2) if is_cuda else 0.0
    )

    summary = {
        "mode": "debug_smoke_test",
        "device": device_name,
        "is_cuda": is_cuda,
        "train_samples": train_samples,
        "eval_samples": eval_samples,
        "epochs": 1.0,
        "max_length": max_length,
        "num_beams": 4,
        "duration_seconds": round(training_duration, 2),
        "peak_vram_mb": peak_vram_mb,
        "train_loss": round(float(train_result.training_loss), 4),
        "eval_loss": debug_loss,
        "eval_bleu": debug_bleu,
        "eval_chrf": debug_chrf,
        "global_step": train_result.global_step,
    }

    with open(output_dir / "train_metrics.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    comparison = {
        "disclaimer": "SMOKE TEST COMPARATIVO - ESTAS MÉTRICAS DEMUESTRAN QUE EL FLUJO DE ENTRENAMIENTO Y EVALUACIÓN FUNCIONA. NO REPRESENTAN EL RESULTADO FINAL DEL PROYECTO.",
        "device": device_name,
        "eval_samples": eval_samples,
        "base_model": {
            "name": model_name,
            "eval_loss": base_loss,
            "bleu": base_bleu,
            "chrf": base_chrf,
        },
        "debug_fine_tuned_model": {
            "name": f"{model_name} (fine-tuned {train_samples} samples, 1 epoch)",
            "eval_loss": debug_loss,
            "bleu": debug_bleu,
            "chrf": debug_chrf,
        },
        "delta": {
            "delta_bleu": round(debug_bleu - base_bleu, 2),
            "delta_chrf": round(debug_chrf - base_chrf, 2),
            "delta_loss": round(debug_loss - base_loss, 4),
        }
    }

    comp_file = project_root / "outputs/smoke_tests/nmt_base_vs_debug.json"
    comp_file.parent.mkdir(parents=True, exist_ok=True)
    with open(comp_file, "w", encoding="utf-8") as f:
        json.dump(comparison, f, indent=2, ensure_ascii=False)

    return summary

# Intentar importar desde módulo local si existe; si no, ejecutar implementación autónoma
try:
    from src.translation.train import run_training
    summary = run_training(debug_mode=True, max_length=128)
except (ImportError, ModuleNotFoundError):
    summary = run_smoke_fine_tuning(project_root=PROJECT_ROOT, max_length=128)

print("Smoke test de fine-tuning completado exitosamente.")

Dispositivo de ejecución: Tesla T4 (CUDA=True)
Smoke test: 256 train, 64 eval, max_length=128, 1 epoch


config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

...

Base Metrics -> Loss: 2.5514 | BLEU: 42.33 | chrF: 64.8
Iniciando entrenamiento del smoke test...


Step,Training Loss,Validation Loss,Bleu,Chrf
16,2.374242,2.561835,40.460000,63.910000
32,2.196053,2.566329,40.930000,64.330000


Entrenamiento completado en 15.76 s
Evaluando modelo FINE-TUNED (smoke test)...


Training Loss,Validation Loss,Step,Bleu,Chrf
2.196053,2.566329,32,40.930000,64.330000


Smoke test de fine-tuning completado exitosamente.


## 3. Resumen de Métricas y Consumo de Recursos

In [4]:
import pandas as pd

df_summary = pd.DataFrame([
    {"Métrica / Parámetro": "Modo", "Valor": summary["mode"]},
    {"Métrica / Parámetro": "Dispositivo", "Valor": summary["device"]},
    {"Métrica / Parámetro": "Muestras Entrenamiento", "Valor": summary["train_samples"]},
    {"Métrica / Parámetro": "Muestras Evaluación", "Valor": summary["eval_samples"]},
    {"Métrica / Parámetro": "Épocas", "Valor": summary["epochs"]},
    {"Métrica / Parámetro": "Longitud Máxima (max_length)", "Valor": summary["max_length"]},
    {"Métrica / Parámetro": "Duración Total (s)", "Valor": summary["duration_seconds"]},
    {"Métrica / Parámetro": "Pico VRAM (MB)", "Valor": summary["peak_vram_mb"]},
    {"Métrica / Parámetro": "Train Loss", "Valor": summary["train_loss"]},
    {"Métrica / Parámetro": "Eval Loss", "Valor": summary["eval_loss"]},
    {"Métrica / Parámetro": "Eval SacreBLEU (Smoke)", "Valor": summary["eval_bleu"]},
    {"Métrica / Parámetro": "Eval chrF (Smoke)", "Valor": summary["eval_chrf"]},
])

display(df_summary)

,Métrica / Parámetro,Valor
0,Modo,debug_smoke_test
1,Dispositivo,Tesla T4
2,Muestras Entrenamiento,256
3,Muestras Evaluación,64
4,Épocas,1.0
5,Longitud Máxima (max_length),128
6,Duración Total (s),15.76
7,Pico VRAM (MB),1669.87
8,Train Loss,2.2612
9,Eval Loss,2.5663


## 4. Comparación: Modelo Base vs. Modelo Debug

In [5]:
import json

comp_path = PROJECT_ROOT / "outputs/smoke_tests/nmt_base_vs_debug.json"
if comp_path.exists():
    with open(comp_path, "r", encoding="utf-8") as f:
        comp = json.load(f)

    print(comp["disclaimer"])
    print("-" * 60)
    df_comp = pd.DataFrame([
        {"Modelo": "Base (Helsinki-NLP/opus-mt-es-en)", "Loss": comp["base_model"][ "eval_loss"], "BLEU": comp["base_model"]["bleu"], "chrF": comp["base_model"]["chrf"]},
        {"Modelo": "Debug Fine-Tuned (256 samples, 1 epoch)", "Loss": comp["debug_fine_tuned_model"]["eval_loss"], "BLEU": comp["debug_fine_tuned_model"]["bleu"], "chrF": comp["debug_fine_tuned_model"]["chrf"]},
    ])
    display(df_comp)
else:
    print("Reporte comparativo no disponible en disco.")

SMOKE TEST COMPARATIVO - ESTAS MÉTRICAS DEMUESTRAN QUE EL FLUJO DE ENTRENAMIENTO Y EVALUACIÓN FUNCIONA. NO REPRESENTAN EL RESULTADO FINAL DEL PROYECTO.
------------------------------------------------------------


,Modelo,Loss,BLEU,chrF
0,Base (Helsinki-NLP/opus-mt-es-en),2.5514,42.33,64.80
1,"Debug Fine-Tuned (256 samples, 1 epoch)",2.5663,40.93,64.33
